# Assignment 5: K-Means and Hierarchical Clustering on Sales Data

**Aim:** Implement K-Means and Hierarchical clustering on `sales_data_sample.csv` and determine a suitable number of clusters using the Elbow Method.

**Dataset:** Download the Kaggle CSV and place it beside this notebook. The notebook attempts customer-level aggregation when a customer identifier is present.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

DATA_PATH = "sales_data_sample.csv"  # Place downloaded CSV beside this notebook

## 2. Load and inspect dataset

In [ ]:
df = pd.read_csv(DATA_PATH, encoding="latin1")
print("Shape:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())
display(df.isnull().sum().sort_values(ascending=False).head(15))

## 3. Preprocess data for clustering
Clustering needs numeric features. This notebook groups sales records by customer when a customer identifier is available, then builds numeric customer-level features. Otherwise it clusters rows using available numeric columns. Identifier-like columns are excluded from the clustering features.

In [ ]:
data = df.copy()
data = data.drop_duplicates()

# Convert date column if present
date_cols = [c for c in data.columns if "date" in c.lower()]
for col in date_cols:
    parsed = pd.to_datetime(data[col], errors="coerce")
    if parsed.notna().mean() > 0.5:
        data[col + "_year"] = parsed.dt.year
        data[col + "_month"] = parsed.dt.month
        data = data.drop(columns=[col])

# Prefer customer-level aggregation for this sales dataset
customer_col = next((c for c in data.columns if c.lower() in ["customername", "customer_name", "customerid", "customer_id"]), None)
if customer_col:
    numeric_cols = data.select_dtypes(include=np.number).columns.tolist()
    if not numeric_cols:
        raise ValueError("No numeric columns available for clustering.")
    cluster_data = data.groupby(customer_col)[numeric_cols].mean().reset_index(drop=True)
    print("Clustering customer-level averages using:", numeric_cols)
else:
    cluster_data = data.select_dtypes(include=np.number).copy()
    print("Customer column not found; clustering rows using numeric columns.")

# Drop constant columns and columns that are likely row identifiers
cluster_data = cluster_data.loc[:, cluster_data.nunique(dropna=True) > 1]
cluster_data = cluster_data.drop(columns=[c for c in cluster_data.columns if c.lower() in ["ordernumber", "order_number", "phone", "postalcode", "postal_code"]], errors="ignore")
cluster_data = cluster_data.replace([np.inf, -np.inf], np.nan)
cluster_data = cluster_data.fillna(cluster_data.median(numeric_only=True))
cluster_data = cluster_data.dropna(axis=1, how="all")

if cluster_data.shape[1] < 1:
    raise ValueError("No usable numeric features found for clustering.")

print("Clustering data shape:", cluster_data.shape)
display(cluster_data.head())

## 4. Standardize features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_data)

# Keep cluster count valid for the available number of records
n_samples = X_scaled.shape[0]
if n_samples < 3:
    raise ValueError("At least 3 records are required for clustering.")
print("Number of records:", n_samples)

## 5. Determine number of clusters using the Elbow Method
The Within-Cluster Sum of Squares (inertia) is calculated for different values of K. The elbow point is a reasonable cluster count where the reduction in inertia begins to slow.

In [ ]:
max_k = min(10, n_samples - 1)
k_values = range(1, max_k + 1)
inertias = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia / WCSS")
plt.title("Elbow Method")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

## 6. Apply K-Means clustering
Choose `N_CLUSTERS` after inspecting the elbow plot. The default is 3 for demonstration; adjust it to the elbow point in your graph.

In [ ]:
N_CLUSTERS = min(3, n_samples - 1)  # Change after examining the elbow plot

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
print("K-Means cluster counts:")
display(pd.Series(kmeans_labels).value_counts().sort_index())
if len(set(kmeans_labels)) > 1:
    print("K-Means silhouette score:", round(silhouette_score(X_scaled, kmeans_labels), 4))

## 7. Apply Hierarchical clustering
Agglomerative clustering is used with the same selected number of clusters. A dendrogram on a sample of at most 500 records is plotted to keep execution manageable.

In [ ]:
hierarchical = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage="ward")
hierarchical_labels = hierarchical.fit_predict(X_scaled)

print("Hierarchical cluster counts:")
display(pd.Series(hierarchical_labels).value_counts().sort_index())
if len(set(hierarchical_labels)) > 1:
    print("Hierarchical silhouette score:", round(silhouette_score(X_scaled, hierarchical_labels), 4))

sample_size = min(500, n_samples)
sample_indices = np.random.RandomState(42).choice(n_samples, size=sample_size, replace=False)
Z = linkage(X_scaled[sample_indices], method="ward")
plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode="lastp", p=30)
plt.title("Hierarchical Clustering Dendrogram (sample)")
plt.xlabel("Sampled data points / merged clusters")
plt.ylabel("Euclidean distance")
plt.show()

## 8. Visualize clusters using the first two scaled features
This is a 2D projection using the first two features, not a complete representation if the dataset has more than two features.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=kmeans_labels, cmap="viridis", s=25)
plt.xlabel("Scaled feature 1: " + str(cluster_data.columns[0]))
plt.ylabel("Scaled feature 2: " + str(cluster_data.columns[1]) if cluster_data.shape[1] > 1 else "Feature 2 unavailable")
plt.title("K-Means Clusters (first two features)")
plt.colorbar(label="Cluster")
plt.show()

## 9. Conclusion
The sales dataset was cleaned and numeric features were standardized before clustering. The Elbow Method was used to inspect a suitable number of clusters. K-Means and Agglomerative Hierarchical Clustering were then implemented and their cluster sizes and silhouette scores were examined. Clusters can be interpreted further using customer/order characteristics in the original dataset.